# 单细胞数据的读取方式总结
单细胞数据（主要是 scRNA-seq，也包括多组学如 scATAC 等）常见的读取形式包括以下几大类。
- 1. 10x Genomics CellRanger 输出（最常见）
- 2. h5ad 格式（AnnData 对象，Python 生态标准）
- 3. Loom 格式
- 4. RDS 格式（Seurat 或 SingleCellExperiment 对象，R 生态标准）
- 5. 纯稀疏矩阵 MTX（Matrix Market）格式
- 6. 表格格式（CSV / TSV / TXT）

这里提供 Python（主要用 Scanpy + AnnData） 和 R（主要用 Seurat） 的读取代码示例。

```python
# 包的安装
# for python
pip install scanpy

# for R
BiocManager::install("Seurat")
```

## 1. 10x Genomics CellRanger 输出（最常见）
格式包括：matrix.mtx、features.tsv.gz/genes.tsv、barcodes.tsv.gz 的目录（MEX/MTX 格式），或单个 .h5 文件。

```r
#######   R 
library(Seurat)

# 读取 MTX 目录（最常用）
counts <- Read10X(data.dir = "path/to/filtered_feature_bc_matrix/")
seurat_obj <- CreateSeuratObject(counts = counts, project = "10x_project")

# 读取 10x h5 文件
counts_h5 <- Read10X_h5(filename = "path/to/filtered_feature_bc_matrix.h5")
seurat_obj <- CreateSeuratObject(counts = counts_h5)

########    拓展
fileID = list.files("./10X_files/") # 存在每个文件夹，文件夹下存在3个文件 barcodes.tsv.gz  features.tsv.gz  matrix.mtx.gz
path = "./10X_files/"

# 运行lapply函数读入数据
seurat.list = lapply(fileID, function(file){
  seurat_data <- Read10X(data.dir = paste0(path, file))
  seurat_obj <- CreateSeuratObject(counts = seurat_data,
                                   min.cells = 0,
                                   min.features = 0,
                                   project = file)
  return(seurat_obj)
})

# 合并seurat对象
merged_seurat <- merge(x = seurat.list[[1]],
                       y = seurat.list[-1],
                       add.cell.id = fileID)
head(merged_seurat@meta.data)
tail(merged_seurat@meta.data)

# 载入表型数据
## 1.3 表型数据载入
merged_seurat$sampleID = merged_seurat$orig.ident

# 根据样本信息重命名编组
merged_seurat$group <- recode(merged_seurat$sampleID,
                              "SRR7722939" = "PBMC_Pre",
                              "SRR7722940" = "PBMC_Disc_Early",
                              "SRR7722941" = "PBMC_Disc_Resp",
                              "SRR7722942" = "PBMC_Disc_AR",
                              "SRR7722937" = "Tumor_Disc_Pre",
                              "SRR7722938" = "Tumor_Disc_AR")
                            
# 保存一下
saveRDS(merged_seurat,file = "./Outdata/Step1.RawCount_merged_seurat.rds")

# tips
# saveRDS 速度慢
system.time({
    saveRDS(merged_seurat,file = "./Outdata/Step1.RawCount_merged_seurat.rds")
           })
#qs速度快
#install.packages('qs')
library(qs)
system.time({
    qsave(merged_seurat,file = "./Outdata/Step1.RawCount_merged_seurat.qs") 
})

# 读取保存的数据
# readRDS 速度慢
# input.data = readRDS(file = "./Outdata/Step1.RawCount_merged_seurat.rds")

# read_rds速度快
library(readr)
system.time({
    input.data = read_rds(file = "./Outdata/Step1.RawCount_merged_seurat.rds")
           })

# qread速度很快
library(qs)
system.time({
    input.data = qread(file = "./Outdata/Step1.RawCount_merged_seurat.qs")
           })
```

```python
###########################   python  #################
import scanpy as sc

# 读取 MTX 目录（推荐）
adata = sc.read_10x_mtx('path/to/filtered_feature_bc_matrix/',  # 或 raw_feature_bc_matrix
                        var_names='gene_symbols',  # 或 'gene_ids'
                        cache=True)

# 读取 10x h5 文件
adata = sc.read_10x_h5('path/to/filtered_feature_bc_matrix.h5')

```


## 2. h5ad 格式（AnnData 对象，Python 生态标准）
格式：HDF5-based，存储表达矩阵 + 元数据 + 降维等。


```python
## python
import scanpy as sc
adata = sc.read_h5ad('path/to/data.h5ad')
```

```R
library(Seurat)
library(SeuratDisk)  # 需要安装 SeuratDisk

# 转换为 h5Seurat 后再读（推荐方式）
Convert("path/to/data.h5ad", dest = "h5seurat", overwrite = TRUE)
seurat_obj <- LoadH5Seurat("path/to/data.h5seurat")

# 或用 zellkonverter 转为 SingleCellExperiment 再转 Seurat（Bioconductor 生态）
library(zellkonverter)
sce <- readH5AD("path/to/data.h5ad")
seurat_obj <- as.Seurat(sce)
```

## 3. Loom 格式
格式：HDF5-based，早期 RNA velocity 常用。



```python
## python
import scanpy as sc
adata = sc.read_loom('path/to/data.loom', sparse=True)  # 或用 loompy 直接读
```

```R
## R 
loom_data <- ReadLoom("path/to/data.loom")  # 或用 loomR::connect
seurat_obj <- as.Seurat(loom_data)
```

## 4. RDS 格式（Seurat 或 SingleCellExperiment 对象，R 生态标准）
格式：R 序列化对象，存储完整 Seurat/SCE 对象



```python
## python
# 推荐用 scDIOR 或 anndataR 等跨语言工具（或 reticulate + rpy2）
# 示例：用 scdior（需安装）
import scdior
adata = scdior.read_rds('path/to/data.rds')  # 转为 AnnData
```

```R
library(Seurat)
seurat_obj <- readRDS("path/to/data.rds")
# 如果是 SingleCellExperiment：
library(SingleCellExperiment)
sce <- readRDS("path/to/data.rds")
seurat_obj <- as.Seurat(sce)
```

## 5. 纯稀疏矩阵 MTX（Matrix Market）格式
格式：单独的 .mtx 文件，常搭配 genes/features 和 barcodes 文件。



```python
## python
import scanpy as sc
adata = sc.read_mtx('path/to/matrix.mtx.gz')  # 需手动设置 obs/var
# 常用：结合 read_10x_mtx 自动处理
```

```R
## R
library(Seurat)
# 通用 MTX 读取
counts <- ReadMtx(mtx = "path/to/matrix.mtx",
                  cells = "path/to/barcodes.tsv",
                  features = "path/to/features.tsv")
seurat_obj <- CreateSeuratObject(counts = counts)
```

## 6. 表格格式（CSV / TSV / TXT）
格式：表达矩阵（基因×细胞或细胞×基因），常用于小型数据集或公开数据。



```python
import scanpy as sc
import pandas as pd

# 读取计数矩阵（假设行为基因，列为细胞）
counts = pd.read_csv('path/to/counts.csv', index_col=0)
adata = sc.AnnData(X=counts.T)  # 转为细胞×基因
# 或直接用 scanpy.read_text / read_csv（旧版）
adata = sc.read_csv('path/to/counts.csv')
```

```R
## R
library(Seurat)
counts <- read.csv("path/to/counts.csv", row.names = 1)  # 或 read.table
seurat_obj <- CreateSeuratObject(counts = as.matrix(counts))  # 或 sparseMatrix
```